## 1. Install

In [ ]:
!pip install -q albumentations opencv-python-headless torchxrayvision optuna \
                scikit-learn matplotlib

## 2. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

## 3. Imports

In [ ]:
import os
import gc
import cv2
import json
import copy
import torch
import optuna
import shutil
import random
import zipfile
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import albumentations as A
import torchxrayvision as xrv

from pathlib import Path
from tqdm.auto import tqdm
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, fbeta_score,
    roc_auc_score, average_precision_score, confusion_matrix,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## 4. Config

Edit `N_TRIALS`, `EPOCHS`, and `save_dir` here. Note the save dir is a **new** folder so your ImageNet results aren't overwritten.

In [ ]:
SEED        = 42
XRV_WEIGHTS = "densenet121-res224-mimic_nb"
EPOCHS      = 25
N_TRIALS    = 25          # raise if you have GPU time
IMG_SIZE    = 224        # XRV densenet is res224; do not change

CONFIG = {
    "dataset_root": "/content/rsna-pneumonia-dataset-npy",
    "model_name":   "DenseNet-121-XRV-mimic_nb",
    "epochs":       EPOCHS,
    "img_size":     IMG_SIZE,
    "save_dir":     "/content/drive/MyDrive/AI_Research/Model_Results/xrv_automated",
}

SAVE_DIR = CONFIG["save_dir"]
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Saving to {SAVE_DIR}")


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def worker_init_fn(worker_id):
    s = torch.initial_seed() % 2**32
    np.random.seed(s)
    random.seed(s)


set_seed(SEED)

## 5. Extract dataset and copy splits

In [ ]:
ZIP_PATH    = "/content/drive/MyDrive/AI_Research/rsna-pneumonia-dataset-npy.zip"
EXTRACT_DIR = "/content/rsna-pneumonia-dataset-npy"
DRIVE_ROOT  = "/content/drive/MyDrive/AI_Research"

if not os.path.exists(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall("/content")
print("Dataset extracted")

for fname in ["train_split.csv", "val_split.csv", "test_split.csv",
              "stage_2_train_labels.csv"]:
    src = os.path.join(DRIVE_ROOT, fname)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(EXTRACT_DIR, fname))
        print(f"Copied {fname}")

## 6. Load the split CSVs

In [ ]:
train_ds = pd.read_csv(f"{EXTRACT_DIR}/train_split.csv")
val_ds   = pd.read_csv(f"{EXTRACT_DIR}/val_split.csv")
test_ds  = pd.read_csv(f"{EXTRACT_DIR}/test_split.csv")

print(f"train {len(train_ds)}   val {len(val_ds)}   test {len(test_ds)}")
print(f"train positives: {int(train_ds['Target'].sum())} "
      f"({train_ds['Target'].mean():.1%})")

## 7. Build / load the NPY map

In [ ]:
NPY_MAP_PATH = f"{DRIVE_ROOT}/npy_map.json"
IMAGES_DIR   = f"{EXTRACT_DIR}/images"

if os.path.exists(NPY_MAP_PATH):
    with open(NPY_MAP_PATH) as f:
        npy_map = json.load(f)
    print("Loaded existing NPY map.")
else:
    print("Building NPY map...")
    npy_map = {f.stem: str(f) for f in Path(IMAGES_DIR).rglob("*.npy")}
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    with open(NPY_MAP_PATH, "w") as f:
        json.dump(npy_map, f)

print(f"{len(npy_map):,} paths")

## 8. Transforms

Same augmentations as your original — Rotate, ShiftScaleRotate, RandomBrightnessContrast, Resize — with `A.Normalize` removed. The XRV rescale happens in the dataset instead.

In [ ]:
# XRV densenets take 1-channel input at roughly [-1024, 1024]:
#     img = (2 * (img / maxval) - 1) * 1024
# No 3-channel stacking, no ImageNet Normalize. Mixing conventions gives ~0.5 AUC.

xrv_train_transform = A.Compose([
    A.Rotate(limit=7, border_mode=cv2.BORDER_CONSTANT, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=0,
                       border_mode=cv2.BORDER_CONSTANT, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.10, contrast_limit=0.10, p=0.5),
    A.Resize(IMG_SIZE, IMG_SIZE),
    ToTensorV2(),
])

xrv_eval_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    ToTensorV2(),
])

## 9. Dataset and collate_fn

In [ ]:
class RSNADatasetXRV(Dataset):
    def __init__(self, dataframe, npy_map, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.npy_map = npy_map
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        pid = row["patientId"]
        if pid not in self.npy_map:
            raise KeyError(f"Missing NPY for {pid}")

        img = np.load(self.npy_map[pid]).astype(np.float32)
        if img.ndim == 3:
            img = img[..., 0]

        img = (img - img.min()) / (img.max() - img.min() + 1e-8)   # -> [0, 1]
        img = img[..., None]                                        # H,W,1

        if self.transform:
            img = self.transform(image=img)["image"]                # 1,H,W

        img = (img * 2.0 - 1.0) * 1024.0                            # -> XRV range

        return {
            "image": img,
            "label": torch.tensor(int(row["Target"]), dtype=torch.long),
            "patient_id": pid,
        }


def collate_fn(batch):
    return {
        "images":      torch.stack([b["image"] for b in batch]),
        "labels":      torch.stack([b["label"] for b in batch]),
        "patient_ids": [b["patient_id"] for b in batch],
    }


train_dataset = RSNADatasetXRV(train_ds, npy_map, xrv_train_transform)
val_dataset   = RSNADatasetXRV(val_ds,   npy_map, xrv_eval_transform)
test_dataset  = RSNADatasetXRV(test_ds,  npy_map, xrv_eval_transform)

print(f"datasets: {len(train_dataset)} / {len(val_dataset)} / {len(test_dataset)}")

## 10. Sanity check — run before training

Expect **1 channel, 224×224, min near −1024, max near 1024**, and chest X-rays that look right way up with dark lungs. If the range is wrong or XRV prints a normalization warning on the first forward pass, stop and fix it — everything downstream will be meaningless.

In [ ]:
_s = train_dataset[0]["image"]
print(f"shape {tuple(_s.shape)}   min {_s.min():.1f}   max {_s.max():.1f}   "
      f"mean {_s.mean():.1f}")
print("want: 1 channel, 224x224, min near -1024, max near 1024")

_b = collate_fn([train_dataset[i] for i in range(4)])
print(f"batch images {tuple(_b['images'].shape)}  labels {_b['labels'].tolist()}")

plt.figure(figsize=(12, 3))
for i in range(4):
    plt.subplot(1, 4, i + 1)
    plt.imshow(_b["images"][i, 0], cmap="gray")
    plt.title(f"label={_b['labels'][i].item()}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 11. Model builder

`op_threshs = None` disables XRV's built-in sigmoid + threshold post-processing, which would otherwise corrupt the logits. The head becomes `Dropout → Linear(1024, 2)` to match the 2-logit `CrossEntropyLoss` setup. `conv0` should print `[64, 1, 7, 7]` — 1 input channel, not 3.

In [ ]:
def build_xrv_model(dropout, weights=XRV_WEIGHTS):
    m = xrv.models.DenseNet(weights=weights)
    m.op_threshs = None                        # disable XRV post-processing
    in_features = m.classifier.in_features      # 1024
    m.classifier = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(in_features, 2),
    )
    return m


_m = build_xrv_model(0.3)
print(f"conv0 {tuple(_m.features.conv0.weight.shape)}  (expect [64, 1, 7, 7])")
print(f"classifier {_m.classifier}")
del _m
gc.collect()

## 12. Class-weighted loss

In [ ]:
counts = np.bincount(train_ds["Target"].values.astype(np.int64))
pos_weight = counts[0] / (counts[1] + 1e-8)
class_weights = torch.tensor([1.0, pos_weight], dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

print(f"class counts {counts.tolist()}   pos_weight {pos_weight:.3f}")

## 13. Evaluation function

In [ ]:
@torch.no_grad()
def evaluate(model, loader, use_hflip=False):
    model.eval()
    probs, labels, losses = [], [], []

    for batch in loader:
        x = batch["images"].to(device, non_blocking=True)
        y = batch["labels"].to(device, dtype=torch.long, non_blocking=True)

        logits = model(x)
        losses.append(criterion(logits, y).item())

        views = [x, torch.flip(x, dims=[-1])] if use_hflip else [x]
        acc = None
        for v in views:
            lg = logits if v is x else model(v)
            p = torch.softmax(lg, dim=1)[:, 1]
            acc = p if acc is None else acc + p

        probs.append((acc / len(views)).float().cpu().numpy())
        labels.append(y.cpu().numpy())

    y_true = np.concatenate(labels)
    y_prob = np.concatenate(probs)
    y_pred = (y_prob >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "loss":        float(np.mean(losses)),
        "accuracy":    float(accuracy_score(y_true, y_pred)),
        "precision":   float(precision_score(y_true, y_pred, zero_division=0)),
        "recall":      float(recall_score(y_true, y_pred, zero_division=0)),
        "specificity": float(tn / (tn + fp + 1e-8)),
        "f1":          float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc":     float(roc_auc_score(y_true, y_prob)),
        "pr_auc":      float(average_precision_score(y_true, y_prob)),
        "fnr":         float(fn / (fn + tp + 1e-8)),
        "y_true":      y_true,
        "y_prob":      y_prob,
    }

## 14. Trial curve plots

Your original three plots, plus a fourth showing the learning rate on a log axis with vertical markers at each unfreezing event.

In [ ]:
def save_trial_curves(history, trial_number):
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss")
    plt.title(f"Trial {trial_number} Loss Curve")
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, f"trial_{trial_number}_loss_curve.png"), dpi=200)
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["val_roc_auc"], label="ROC-AUC")
    plt.plot(epochs, history["val_pr_auc"], label="PR-AUC")
    plt.xlabel("Epoch"); plt.ylabel("AUC")
    plt.title(f"Trial {trial_number} AUC Curve")
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, f"trial_{trial_number}_auc_curve.png"), dpi=200)
    plt.close()

    plt.figure(figsize=(8, 5))
    for k in ["accuracy", "precision", "recall", "f1"]:
        plt.plot(epochs, history[f"val_{k}"], label=k.capitalize())
    plt.xlabel("Epoch"); plt.ylabel("Score")
    plt.title(f"Trial {trial_number} Classification Metrics")
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR,
                             f"trial_{trial_number}_classification_curve.png"), dpi=200)
    plt.close()

    # learning rate + unfreezing schedule
    fig, ax1 = plt.subplots(figsize=(8, 5))
    ax1.plot(epochs, history["learning_rate"], color="tab:blue", label="LR")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Learning Rate", color="tab:blue")
    ax1.set_yscale("log"); ax1.grid(True, alpha=0.3)
    seen = set()
    for e, fs in zip(epochs, history["freeze_state"]):
        if fs not in seen:
            seen.add(fs)
            ax1.axvline(e, ls="--", alpha=0.5, color="tab:red")
            ax1.text(e, ax1.get_ylim()[1], f" {fs}", rotation=90,
                     va="top", fontsize=8, color="tab:red")
    plt.title(f"Trial {trial_number} LR + Unfreezing")
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, f"trial_{trial_number}_lr_schedule.png"), dpi=200)
    plt.close()

## 15. Optuna objective

In [ ]:
def objective(trial):
    set_seed(SEED)                       # every trial starts identically

    history = {"train_loss": [], "val_loss": [], "val_accuracy": [],
               "val_precision": [], "val_recall": [], "val_specificity": [],
               "val_f1": [], "val_roc_auc": [], "val_pr_auc": [],
               "learning_rate": [], "freeze_state": [], "trainable_params": []}

    lr             = trial.suggest_float("initial_lr", 1e-5, 1e-3, log=True)
    weight_decay   = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    dropout        = trial.suggest_float("dropout", 0.2, 0.5)
    batch_size     = trial.suggest_categorical("batch_size", [64, 128])
    optimizer_name = trial.suggest_categorical("optimizer", ["AdamW", "Adam"])
    strategy       = trial.suggest_categorical(
        "strategy",
        ["denseblock4_then_full", "denseblock4_only", "full_backbone", "frozen"])
    scheduler_name = trial.suggest_categorical(
        "scheduler", ["cosine", "step", "plateau"])

    if strategy in ("denseblock4_then_full", "denseblock4_only"):
        block4_epoch  = trial.suggest_int("block4_epoch", 2, 5)
        block4_factor = trial.suggest_float("block4_factor", 0.1, 0.4)
    else:
        block4_epoch = block4_factor = None

    if strategy == "denseblock4_then_full":
        full_epoch  = trial.suggest_int("full_epoch", 6, 12)
        full_factor = trial.suggest_float("full_factor", 0.3, 0.8)
    else:
        full_epoch = full_factor = None

    # model
    model = build_xrv_model(dropout).to(device)
    for p in model.features.parameters():
        p.requires_grad = False
    if strategy == "full_backbone":
        for p in model.features.parameters():
            p.requires_grad = True

    #  optimizer
    opt_cls = optim.AdamW if optimizer_name == "AdamW" else optim.Adam
    optimizer = opt_cls(model.parameters(), lr=lr, weight_decay=weight_decay)

    # scheduler
    if scheduler_name == "cosine":
        cosine_tmax = trial.suggest_int("cosine_tmax", 10, EPOCHS)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=cosine_tmax)
        scheduler_params = {"cosine_tmax": cosine_tmax}
    elif scheduler_name == "step":
        step_size  = trial.suggest_int("step_size", 3, 8)
        step_gamma = trial.suggest_float("step_gamma", 0.1, 0.5)
        scheduler = torch.optim.lr_scheduler.StepLR(
            optimizer, step_size=step_size, gamma=step_gamma)
        scheduler_params = {"step_size": step_size, "step_gamma": step_gamma}
    else:
        plateau_factor   = trial.suggest_float("plateau_factor", 0.2, 0.7)
        plateau_patience = trial.suggest_int("plateau_patience", 1, 3)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", factor=plateau_factor, patience=plateau_patience)
        scheduler_params = {"plateau_factor": plateau_factor,
                            "plateau_patience": plateau_patience}

    # loaders
    g = torch.Generator()
    g.manual_seed(SEED)

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, generator=g,
        collate_fn=collate_fn, num_workers=8, pin_memory=True,
        worker_init_fn=worker_init_fn, drop_last=True)
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        collate_fn=collate_fn, num_workers=8, pin_memory=True,
        worker_init_fn=worker_init_fn)

    amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    scaler = torch.amp.GradScaler("cuda", enabled=(amp_dtype == torch.float16))

    cfg = {"trial": trial.number, "seed": SEED, "epochs": EPOCHS,
           "architecture": "DenseNet121", "pretrained": XRV_WEIGHTS,
           "initial_lr": lr, "weight_decay": weight_decay, "dropout": dropout,
           "batch_size": batch_size, "optimizer": optimizer_name,
           "strategy": strategy, "scheduler": scheduler_name,
           "block4_epoch": block4_epoch, "block4_factor": block4_factor,
           "full_epoch": full_epoch, "full_factor": full_factor,
           **scheduler_params}

    with open(os.path.join(SAVE_DIR,
                           f"trial_{trial.number}_hyperparams.json"), "w") as f:
        json.dump(cfg, f, indent=4)

    print(f"\n{'='*70}\nTrial {trial.number}\n{json.dumps(cfg, indent=2)}\n{'='*70}")

    best_val_auc, best_epoch, best_state = -1.0, -1, None

    try:
        for epoch in range(1, EPOCHS + 1):

            # unfreezing schedule
            if strategy in ("denseblock4_then_full", "denseblock4_only") \
               and epoch == block4_epoch:
                for p in model.features.denseblock4.parameters():
                    p.requires_grad = True
                for p in model.features.norm5.parameters():
                    p.requires_grad = True
                new_lr = lr * block4_factor
                for grp in optimizer.param_groups:
                    grp["lr"] = new_lr
                print(f"  Stage 1: denseblock4 + norm5 unfrozen, lr={new_lr:.2e}")

            if strategy == "denseblock4_then_full" and epoch == full_epoch:
                for p in model.features.parameters():
                    p.requires_grad = True
                new_lr = lr * block4_factor * full_factor
                for grp in optimizer.param_groups:
                    grp["lr"] = new_lr
                print(f"  Stage 2: full backbone unfrozen, lr={new_lr:.2e}")

            # train
            model.train()
            running = 0.0
            pbar = tqdm(train_loader, desc=f"trial {trial.number} ep {epoch}/{EPOCHS}",
                        leave=False)
            for batch in pbar:
                x = batch["images"].to(device, non_blocking=True)
                y = batch["labels"].to(device, dtype=torch.long, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda", dtype=amp_dtype):
                    loss = criterion(model(x), y)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                running += loss.item()
                pbar.set_postfix(loss=f"{loss.item():.4f}")

            train_loss = running / max(len(train_loader), 1)

            # validate
            vm = evaluate(model, val_loader)
            current_lr = optimizer.param_groups[0]["lr"]

            if strategy == "full_backbone":
                fs = "full_backbone"
            elif strategy == "frozen":
                fs = "frozen"
            elif strategy == "denseblock4_then_full" and epoch >= full_epoch:
                fs = "full_backbone"
            elif block4_epoch is not None and epoch >= block4_epoch:
                fs = "denseblock4+norm5"
            else:
                fs = "frozen"

            history["train_loss"].append(float(train_loss))
            history["val_loss"].append(float(vm["loss"]))
            for k in ["accuracy", "precision", "recall", "specificity",
                      "f1", "roc_auc", "pr_auc"]:
                history[f"val_{k}"].append(float(vm[k]))
            history["learning_rate"].append(float(current_lr))
            history["freeze_state"].append(fs)
            history["trainable_params"].append(
                int(sum(p.numel() for p in model.parameters() if p.requires_grad)))

            print(f"  ep {epoch:>2}  loss {train_loss:.4f}  "
                  f"auc {vm['roc_auc']:.4f}  ap {vm['pr_auc']:.4f}  "
                  f"rec {vm['recall']:.4f}  prec {vm['precision']:.4f}  "
                  f"lr {current_lr:.2e}  [{fs}]")

            with open(os.path.join(SAVE_DIR,
                                   f"trial_{trial.number}_history.json"), "w") as f:
                json.dump(history, f, indent=4)
            save_trial_curves(history, trial.number)

            # save best trial
            if vm["roc_auc"] > best_val_auc:
                best_val_auc = float(vm["roc_auc"])
                best_epoch = epoch
                best_state = copy.deepcopy(model.state_dict())

                torch.save(best_state,
                           os.path.join(SAVE_DIR,
                                        f"trial_{trial.number}_model.pth"))

                with open(os.path.join(
                        SAVE_DIR,
                        f"trial_{trial.number}_best_metrics.json"), "w") as f:
                    json.dump({"trial": trial.number, "best_epoch": best_epoch,
                               "seed": SEED, "pretrained": XRV_WEIGHTS,
                               "freeze_state": fs,
                               "val_loss": vm["loss"],
                               "val_accuracy": vm["accuracy"],
                               "val_precision": vm["precision"],
                               "val_recall": vm["recall"],
                               "val_specificity": vm["specificity"],
                               "val_f1": vm["f1"],
                               "val_roc_auc": vm["roc_auc"],
                               "val_pr_auc": vm["pr_auc"],
                               "val_fnr": vm["fnr"],
                               "learning_rate": float(current_lr)}, f, indent=4)

                print(f"  -> new best (epoch {best_epoch}, auc {best_val_auc:.4f})")

            # prune
            trial.report(vm["roc_auc"], epoch)
            if trial.should_prune():
                print(f"  Trial {trial.number} pruned at epoch {epoch}")
                raise optuna.exceptions.TrialPruned()

            if scheduler_name == "plateau":
                scheduler.step(vm["roc_auc"])
            else:
                scheduler.step()

    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()

    print(f"\nTrial {trial.number} complete. "
          f"Best epoch {best_epoch}, best val AUC {best_val_auc:.4f}")
    return best_val_auc

In [ ]:
NPY_MAP_PATH = f"{DRIVE_ROOT}/npy_map.json"
IMAGES_DIR   = f"{EXTRACT_DIR}/images"      # local: /content/...

def build_map():
    m = {f.stem: str(f) for f in Path(IMAGES_DIR).rglob("*.npy")}
    if not m:
        raise RuntimeError(f"No .npy files under {IMAGES_DIR} — check the extract")
    return m

if os.path.exists(NPY_MAP_PATH):
    with open(NPY_MAP_PATH) as f:
        npy_map = json.load(f)
    sample = next(iter(npy_map.values()))
    # cached map may point at Drive or an old location -- verify before trusting
    if "/content/drive" in sample or not os.path.exists(sample):
        print(f"Cached map is stale ({sample}) -- rebuilding from local disk")
        npy_map = build_map()
        with open(NPY_MAP_PATH, "w") as f:
            json.dump(npy_map, f)
    else:
        print("Loaded cached NPY map.")
else:
    print("Building NPY map...")
    npy_map = build_map()
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    with open(NPY_MAP_PATH, "w") as f:
        json.dump(npy_map, f)

sample = next(iter(npy_map.values()))
print(f"{len(npy_map):,} paths")
print(f"sample: {sample}")
print(f"local disk: {'/content/drive' not in sample}")   # want True

In [ ]:
print(json.dumps(study.trials[-1].params, indent=2))

In [ ]:
import os, subprocess, time
print("CPUs:", os.cpu_count())
print(subprocess.run(["nvidia-smi","--query-gpu=utilization.gpu,memory.used",
                      "--format=csv"], capture_output=True, text=True).stdout)

# pure data loading, no model
loader = DataLoader(train_dataset, batch_size=64, shuffle=True,
                    collate_fn=collate_fn, num_workers=2, pin_memory=True)
t = time.time()
for i, _ in enumerate(loader):
    if i == 30: break
print(f"data only: {(time.time()-t)/31*1000:.0f} ms/batch")

## 16. Run the study

25 trials × 25 epochs is long on a T4. The pruner kills weak trials after epoch 10.

In [ ]:
EPOCHS = 25
print(EPOCHS)

In [ ]:
import os, json
import optuna
from optuna.trial import TrialState, create_trial
from optuna.distributions import (
    FloatDistribution, IntDistribution, CategoricalDistribution)

torch.backends.cudnn.benchmark = True

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=2, n_warmup_steps=10),
)

DISTS = {
    "initial_lr":       FloatDistribution(1e-5, 1e-3, log=True),
    "weight_decay":     FloatDistribution(1e-6, 1e-3, log=True),
    "dropout":          FloatDistribution(0.2, 0.5),
    "batch_size":       CategoricalDistribution([64, 128]),
    "optimizer":        CategoricalDistribution(["AdamW", "Adam"]),
    "strategy":         CategoricalDistribution(
        ["denseblock4_then_full", "denseblock4_only", "full_backbone", "frozen"]),
    "scheduler":        CategoricalDistribution(["cosine", "step", "plateau"]),
    "block4_epoch":     IntDistribution(2, 5),
    "block4_factor":    FloatDistribution(0.1, 0.4),
    "full_epoch":       IntDistribution(6, 12),
    "full_factor":      FloatDistribution(0.3, 0.8),
    "cosine_tmax":      IntDistribution(10, EPOCHS),
    "step_size":        IntDistribution(3, 8),
    "step_gamma":       FloatDistribution(0.1, 0.5),
    "plateau_factor":   FloatDistribution(0.2, 0.7),
    "plateau_patience": IntDistribution(1, 3),
}

SEARCH_DIRS = ["/content", SAVE_DIR]
restored = 0

for n in [1, 2]:
    hp_file = hist_file = None
    for d in SEARCH_DIRS:
        p1 = os.path.join(d, f"trial_{n}_hyperparams.json")
        p2 = os.path.join(d, f"trial_{n}_history.json")
        if os.path.exists(p1) and os.path.exists(p2):
            hp_file, hist_file = p1, p2
            break

    if hp_file is None:
        print(f"  trial {n}: not found, skipping")
        continue

    cfg = json.load(open(hp_file))
    cfg = cfg.get("params", cfg)              # handles both file shapes
    hist = json.load(open(hist_file))

    aucs = hist["val_roc_auc"]
    params = {k: v for k, v in cfg.items() if k in DISTS and v is not None}

    study.add_trial(create_trial(
        state=TrialState.COMPLETE,
        params=params,
        distributions={k: DISTS[k] for k in params},
        value=float(max(aucs)),
        intermediate_values={i + 1: float(a) for i, a in enumerate(aucs)},
    ))
    restored += 1
    print(f"  trial {n}: {len(aucs)} epochs, best auc {max(aucs):.4f}")

print(f"\nRestored {restored} trials. Pruning active from the next trial.\n")

study.optimize(objective, n_trials=N_TRIALS)

print(f"\nBest trial:  {study.best_trial.number}")
print(f"Best AUC:    {study.best_value:.4f}")
print(f"Best params: {json.dumps(study.best_trial.params, indent=2)}")

failsafe

In [ ]:
# worker failsafe
import psutil

def safe_workers(batch_size):
    """Scale workers to available RAM and CPU count."""
    cpus = os.cpu_count() or 2
    gb = psutil.virtual_memory().total / 1e9
    w = min(cpus - 1, 6)
    if batch_size >= 128:
        w = min(w, 4)          # big batches need more RAM per worker
    if gb < 30:
        w = min(w, 2)
    return max(w, 0)


def make_loader(dataset, batch_size, shuffle, generator=None, workers=None):
    w = safe_workers(batch_size) if workers is None else workers
    kw = dict(batch_size=batch_size, shuffle=shuffle, collate_fn=collate_fn,
              pin_memory=True, num_workers=w, drop_last=shuffle)
    if generator is not None:
        kw["generator"] = generator
    if w > 0:
        kw.update(worker_init_fn=worker_init_fn, persistent_workers=True,
                  prefetch_factor=2)
    return DataLoader(dataset, **kw)


def loader_with_retry(dataset, batch_size, shuffle, generator=None):
    """Fall back to fewer workers if they get OOM-killed."""
    for w in [safe_workers(batch_size), 2, 0]:
        try:
            dl = make_loader(dataset, batch_size, shuffle, generator, workers=w)
            next(iter(dl))                       # force worker spawn now
            print(f"  [loader] num_workers={w}")
            return dl
        except RuntimeError as e:
            if "worker" not in str(e).lower():
                raise
            print(f"  [loader] {w} workers failed, retrying lower")
    raise RuntimeError("could not build a working DataLoader")

## 17. Persist study summary

In [ ]:
for t in study.trials:
    with open(os.path.join(SAVE_DIR,
                           f"trial_{t.number}_hyperparams.json"), "w") as f:
        json.dump({"params": t.params, "value": t.value,
                   "state": str(t.state)}, f, indent=4)

with open(os.path.join(SAVE_DIR, "study_summary.json"), "w") as f:
    json.dump({"pretrained": XRV_WEIGHTS, "seed": SEED, "epochs": EPOCHS,
               "n_trials": N_TRIALS,
               "best_trial": study.best_trial.number,
               "best_value": study.best_value,
               "best_params": study.best_trial.params,
               "all_trials": [{"number": t.number, "value": t.value,
                               "state": str(t.state), "params": t.params}
                              for t in study.trials]}, f, indent=4)

print(pd.DataFrame([{"trial": t.number, "val_auc": t.value, "state": str(t.state)}
                    for t in study.trials]).to_string(index=False))

## 18. Load the best model

In [ ]:
BEST_TRIAL = study.best_trial.number
best_params = study.best_trial.params

best_model = build_xrv_model(best_params["dropout"]).to(device)
best_model.load_state_dict(torch.load(
    os.path.join(SAVE_DIR, f"trial_{BEST_TRIAL}_model.pth"), map_location=device))
best_model.eval()
print(f"Loaded trial_{BEST_TRIAL}_model.pth")

## 19. Final test evaluation

Threshold tuned for best F2 on **validation only**, then test scored exactly once.

Note that `trial_N_best_metrics.json` records validation metrics at the default 0.5 threshold, while this cell uses the tuned threshold — don't put those recall numbers in the same table without noting the different operating points.

In [ ]:
bs = best_params["batch_size"]

val_loader_final = DataLoader(val_dataset, batch_size=bs, shuffle=False,
                              collate_fn=collate_fn, num_workers=8, pin_memory=True)
test_loader_final = DataLoader(test_dataset, batch_size=bs, shuffle=False,
                               collate_fn=collate_fn, num_workers=8, pin_memory=True)

vm = evaluate(best_model, val_loader_final, use_hflip=True)

grid = np.linspace(0.01, 0.99, 99)
f2s = [fbeta_score(vm["y_true"], (vm["y_prob"] >= t).astype(int),
                   beta=2, zero_division=0) for t in grid]
best_threshold = float(grid[int(np.argmax(f2s))])

print(f"Tuned threshold (best F2, validation): {best_threshold:.4f}")
print(f"Validation ROC-AUC: {vm['roc_auc']:.4f}")

tm = evaluate(best_model, test_loader_final, use_hflip=True)
y_true, y_prob = tm["y_true"], tm["y_prob"]
y_pred = (y_prob >= best_threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

final_results = {
    "architecture": "DenseNet121",
    "pretrained": XRV_WEIGHTS,
    "seed": SEED,
    "best_trial": BEST_TRIAL,
    "best_params": best_params,
    "threshold_used": best_threshold,
    "val_roc_auc": float(vm["roc_auc"]),
    "test_roc_auc": float(roc_auc_score(y_true, y_prob)),
    "test_pr_auc": float(average_precision_score(y_true, y_prob)),
    "test_accuracy": float(accuracy_score(y_true, y_pred)),
    "test_precision": float(precision_score(y_true, y_pred, zero_division=0)),
    "test_recall": float(recall_score(y_true, y_pred, zero_division=0)),
    "test_specificity": float(tn / (tn + fp + 1e-8)),
    "test_f1": float(f1_score(y_true, y_pred, zero_division=0)),
    "test_f2": float(fbeta_score(y_true, y_pred, beta=2, zero_division=0)),
    "confusion_matrix": [[int(tn), int(fp)], [int(fn), int(tp)]],
}

print(json.dumps(final_results, indent=4))

with open(os.path.join(SAVE_DIR, "final_test_results.json"), "w") as f:
    json.dump(final_results, f, indent=4)

print(f"\nSaved -> {os.path.join(SAVE_DIR, 'final_test_results.json')}")